In [0]:
from datetime import datetime

from pyspark.sql import functions as F

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType

 

LANDING_PATH = "/Volumes/automotive_catalog/automotive_mfg/landing/supplier_delivery"

ARCHIVE_PATH = "/Volumes/automotive_catalog/automotive_mfg/archive/supplier_delivery"

BRONZE_TABLE = "automotive_catalog.automotive_mfg.bronze_supplier_delivery"

 

def audit_log(message):

    print(f"[AUDIT] {message}")

In [0]:
schema = StructType([

    StructField("delivery_id", StringType(), True),

    StructField("supplier_id", StringType(), True),

    StructField("part_number", StringType(), True),

    StructField("part_description", StringType(), True),

    StructField("plant_code", StringType(), True),

    StructField("planned_qty", IntegerType(), True),

    StructField("delivered_qty", IntegerType(), True),

    StructField("delivery_status", StringType(), True),

    StructField("delivery_date", DateType(), True),

    StructField("op_code", StringType(), True)

])

 

batch_run_id = f"batch_{datetime.utcnow().strftime('%Y%m%d%H%M%S')}"

 

# Landing / archive folders (no-op if they already exist)

dbutils.fs.mkdirs(LANDING_PATH)

dbutils.fs.mkdirs(ARCHIVE_PATH)

 

landing_files = [

    f.path for f in dbutils.fs.ls(LANDING_PATH)

    if f.name.lower().endswith(".csv")

]

In [0]:
if not landing_files:

    audit_log("No new supplier_delivery files in landing - nothing to ingest")

else:

    audit_log(f"Batch {batch_run_id}: found {len(landing_files)} file(s) in landing")

 

    df = (

        spark.read

        .option("header", True)

        .schema(schema)

        .csv(LANDING_PATH)

        .withColumn("ingest_ts", F.current_timestamp())

        .withColumn("source_file_name", F.col("_metadata.file_path"))

        .withColumn("batch_run_id", F.lit(batch_run_id))

        .withColumn("load_date", F.current_date())

    )

 

    row_count = df.count()

    df.write.format("delta").mode("append").saveAsTable(BRONZE_TABLE)

    audit_log(f"Loaded {row_count} rows into {BRONZE_TABLE}")

In [0]:
# Archive only after a successful write - this is what makes

# the file ingestion exactly-once per file.

for file_path in landing_files:

 file_name = file_path.rstrip("/").split("/")[-1]

 dbutils.fs.mv(file_path, f"{ARCHIVE_PATH}/{file_name}")

audit_log(f"Moved {len(landing_files)} file(s) to {ARCHIVE_PATH}")

In [0]:
%sql

select * from automotive_catalog.automotive_mfg.bronze_supplier_delivery